大模型在通过 SFT 和对齐（DPO/RLHF）获得听从人类指令、具备良好价值观的能力后，仍然存在一个核心局限：它是一个静态的知识库，无法感知实时互联网，无法主动执行任务，更无法进行多步骤的复杂推理。

要打破大模型的“空谈”局限，让其从“只会生成的语言模型”进化为“能够解决具体问题的数字实体”，我们就必须引入 Agent（智能体） 框架。

ReAct（Reason + Act，推理与行动）框架是解密 Agent 推理的核心关键。

## ReAct 协同推理机制
1. 为什么“只推理不行动（CoT）”或“只行动不推理（Act）”都走不通，而 ReAct 能实现 1+1 > 2 的效果？
2. 在 PyTorch/Python 中，如何徒手实现一个无需复杂框架（如 LangChain）的 ReAct Agent 自循环控制引擎？


#### ReAct 框架的化学反应
在 ReAct 提出之前，让大模型解决复杂问题主要有两种流派：
* CoT (Chain of Thought, 思维链)：让大模型“一步一步地思考”。它极大地提升了逻辑推理能力，但模型依然无法与外部世界交互（无法查数据库、调 API、搜索网页）。
* Act-Only (仅行动)：直接让模型输出要调用的工具（Tool Use）。但由于缺乏中间推理，模型在面对稍微复杂的任务时极易迷失方向，导致“乱调工具”。

斯坦福与谷歌团队提出的 ReAct 框架，将推理（Reasoning）和行动（Acting）紧密结合。它通过一个经典的自循环结构展开：
$$\text{Thought} \rightarrow \text{Action} \rightarrow \text{Observation} \rightarrow \text{Thought} \dots$$

ReAct 的标准四步循环
```
[Thought] (思考): "用户问2026年世界杯在哪里举办。我的内置知识只到2025年，我需要搜索网页。"
   ↓
[Action] (行动): 调用搜索引擎，传入参数 "2026 World Cup host countries"。
   ↓
[Observation] (观察): 搜索引擎返回结果："2026年世界杯由美国、加拿大和墨西哥联合举办。"
   ↓
[Thought] (思考): "我已经得到了答案。现在我可以整理并回答用户了。"
```
这种“思考-行动-观察”的交替运行，让模型在每一步行动后都能看一眼“现实世界的反馈”（Observation），并根据反馈调整接下来的思考，从而极大地降低了幻觉，提升了复杂任务的成功率。



#### 实现 ReAct Agent 自循环引擎
很多同学觉得 Agent 很高深，其实它的底层原理就是一个由 Prompt 驱动的 `while` 循环。

下面我们不依赖任何第三方 Agent 库，用最底层的 Python 代码，手动构建一个能够计算复杂数学公式、并能主动调用工具的 ReAct 引擎。

In [ ]:
import re

# 1. 定义工具库 (Tools)
# 真实的 Agent 会通过 API 调用天气、搜索等，这里我们模拟两个基础工具
def calculate(expression: str) -> str:
    """计算数学表达式"""
    try:
        # 安全地评估数学表达式
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            return str(eval(expression))
        return "Error: 包含非法字符"
    except Exception as e:
        return f"Error: {str(e)}"

def get_current_date(fmt=None) -> str:
    """获取当前日期"""
    return "2026-07-17" # 模拟当前真实时间

# 工具映射表
TOOLS = {
    "calculate": calculate,
    "get_current_date": get_current_date
}

# 2. 构造 ReAct 系统提示词 (System Prompt)
# 这一步是灵魂：它通过 Few-Shot 强行给大模型注入“思考-行动-观察”的复读机逻辑
SYSTEM_PROMPT = """你是一个具备推理和行动能力的 AI 助手。你可以使用以下工具来回答用户的问题：

1. calculate[expression]: 输入一个数学表达式，返回计算结果。
2. get_current_date[]: 无输入，返回当前日期。

你必须严格按照以下格式进行输出，不要输出任何多余的信息：

Thought: 你当前对问题的思考，分析下一步需要做什么。
Action: 工具名称[工具输入] (例如: calculate[2+2] 或 get_current_date[])
Observation: 工具返回的结果（这部分由系统自动输入，你不需要自己生成）

当你想到了最终答案时，请以以下格式结束输出：
Final Answer: 你的最终回答。

现在开始！
"""

# 3. 模拟大模型 (LLM) 行为
# 在真实场景中，这里应该调用 OpenAI API 或本地 Llama 模型
# 这里我们用硬编码的模拟输出，来展示 ReAct 在多轮交互中的 prompt 拼接和状态流转
class MockLLM:
    def __init__(self):
        self.step = 0

    def generate(self, full_prompt: str) -> str:
        """根据当前完整的历史 Prompt，模拟大模型的单步输出"""
        self.step += 1
        if self.step == 1:
            # 第一步：模型需要先获取日期，以便知道离 2030 年还有多久
            return "Thought: 用户想知道距离2030年元旦还有多少天。我需要先获取今天的日期。\nAction: get_current_date[]"
        elif self.step == 2:
            # 第二步：模型拿到了日期 (2026-07-17)，需要计算离 2030-01-01 的天数差
            # 简化计算：这里模型决定计算大体天数 (3.5年 * 365天)
            return """Thought: 今天的日期是 2026-07-17。
            从现在到 2030-01-01 大约还有 3.5 年。
            我需要精确计算 2026年剩余天数 + 3整年 + 2030年前置天数。
            或者我直接计算粗略的公式 3.45 * 365。\nAction: calculate[3.45 * 365]"""
        else:
            # 第三步：模型拿到计算结果，给出最终答案
            return "Thought: 得到了大概的计算结果 1259。现在我可以给出最终答案了。\nFinal Answer: 距离2030年元旦大约还有 1259 天。"

# 4. ReAct 核心控制循环 (Controller Loop)
def run_react_agent(user_query: str):
    llm = MockLLM()
    # 历史记录初始化，拼接上系统 prompt 和用户的提问
    session_history = SYSTEM_PROMPT + f"\nUser: {user_query}\n"

    max_steps = 5
    print(f"🚀 【Agent 启动】收到任务: {user_query}\n")

    for i in range(max_steps):
        print(f"--- 循环第 {i+1} 步 ---")

        # 1. 调用大模型，生成当前步骤 (模型只会输出到 Action：为止，等待 Observation)
        llm_output = llm.generate(session_history)
        print(llm_output)

        # 将模型的输出追加到历史记录中
        session_history += llm_output + "\n"

        # 2. 解析模型的输出，检查是否包含 Final Answer
        if "Final Answer:" in llm_output:
            final_ans = llm_output.split("Final Answer:")[-1].strip()
            print(f"\n🎯 【任务达成】Final Answer: {final_ans}")
            break

        # 3. 正则匹配 Action 提取工具和参数
        action_match = re.search(r"Action:\s*(\w+)\[(.*?)\]", llm_output)
        if action_match:
            tool_name = action_match.group(1)
            tool_input = action_match.group(2)

            # 4. 执行行动并拿到观察结果 (Observation)
            if tool_name in TOOLS:
                print(f"🔧 [执行工具] 调用 {tool_name}，参数: {tool_input}")
                observation = TOOLS[tool_name](tool_input)
            else:
                observation = f"Error: 找不到工具 {tool_name}"

            print(f"👁️ [获得观察] Observation: {observation}\n")

            # 5. 将 Observation 追加回 Prompt，作为下一次大模型推理的上下文
            session_history += f"Observation: {observation}\n"
        else:
            print("⚠️ 未检测到有效的 Action 格式，强行终止。")
            break

if __name__ == "__main__":
    run_react_agent("今天距离2030年元旦还有多少天？")

1. 直面 ReAct 的“死循环”与“解析崩溃”瓶颈：
    在上面的 `run_react_agent` 核心控制循环中，我们使用正则表达式 `r"Action:\s*(\w+)\[(.*?)\]"` 来解析模型的输出。
   * 工程隐患：如果大模型在前向生成时，由于温度（Temperature）过高或者底座能力不足，稍微手抖了一下，输出了 `Action: calculate(3.45 * 365)`（把方括号写成了圆括号），或者漏掉了 `Action:` 前缀。
      * 此时我们的控制循环会发生什么？
   * 思考：在真实的 Agent 工程（如 LangChain、LlamaIndex）中，为了防止模型因为“输出格式微调”而导致整个程序 Crash，通常会采取哪些容错设计（提示：Parser 鲁棒性、LLM 报错重试、JsonMode 约束）？

2. 多步骤规划与短时记忆（Working Memory）的丢失：
    观察我们拼接的 `session_history`：每一次循环，我们都会把 `Thought -> Action -> Observation` 滚雪球一样不断往后拼接，然后重新作为输入送给大模型。
   * 随着 Agent 解决的问题越来越复杂，步骤达到了 20 步甚至 50 步，大模型的 Context Window（上下文窗口） 很快就会被耗尽，且首字延迟（TTFT）会越来越高。
   * 架构设计：为了让 Agent 能够执行长达数小时的超长任务，你认为应该如何对 `session_history` 进行管理或“修剪”（Memory Management）？怎样才能做到既不让模型遗忘前文的核心结论，又不会撑爆显存？

## Agent 的自愈解析与动态记忆管理
针对上面的ReAct 自循环中的“格式解析崩溃”与“死循环”瓶颈，及上下文问题。
那么Agent需具备自愈能力（Self-Healing Parser）和动态滑窗记忆管理的能力。

1. 自愈解析（Self-Healing Parser）：
    * 当 LLM 输出的工具调用格式损坏（如把 `calculate[2+2]` 写成 `calculate(2+2)`）时，Agent 如何不崩溃，而是通过规则修正或自我反思（Self-Correction）实现容错。
2. 动态滑动窗口记忆（Dynamic Sliding Window Memory）：
    * 如何设计一个记忆管理器，在维持历史关键决策（Thought/Action）的同时，自动修剪无关、冗余的工具返回数据（Observation），确保 Context 不超限。

#### Agent 工业落地的三大“隐形炸弹”
在玩具级别的 Demo 中，Agent 看起来很完美。但在生产环境中，有三大“隐形炸弹”会导致 Agent 频繁挂掉：

1. 结构化输出的脆弱性 (The Fragility of Structure)
    LLM 本质上是概率抽样生成下一个 Token。即使你在 System Prompt 里千叮咛万嘱咐“只能输出 `Action: tool_name[arg]`”，模型在 `Temperature > 0` 时依然有高概率输出不合规的格式。一旦 Regex 解析失败，代码抛出 `AttributeError`，整个业务流就会中断。

2. 观察值暴涨 (Observation Explosion)
    当 Agent 调用搜索引擎、数据库查询或读取大文件时，工具返回的 `Observation` 可能长达数千字。如果原封不动地把这些 `Observation` 塞进下一次迭代的 Prompt 中，上下文窗口（Context Window）会呈指数级积压，导致：
   * 推理成本（API Token 消耗）暴增
   * 显存爆炸（OOM） 或触及模型最大 Context 限制。
   * 注意力涣散（LLM 在超长上下文中容易遗忘关键指令，即 Lost in the Middle 现象）。

3. 规划失控与幻觉死循环 (The Infinite Loop)
    当工具返回的结果不符合预期时，模型可能会陷入死循环：不断尝试同一个错误工具，或者在相同的 Thought 中打转。我们需要设定硬性的最大步数限制（Max Iterations），并在格式损坏时，将错误日志作为 `Observation` 反馈给模型，引导其自行修正（Self-Correction）。



#### 编写一个高可用的自愈型 Agent 引擎
1. 容错解析器（支持多种格式提取，并在解析失败时，生成“报错提示”塞回上下文，让模型在下一步自我修复）。
2. 动态记忆管理器（限制保留的历史轮数，并对长工具输出进行截断或摘要）。


In [ ]:
import re
from typing import Dict, Any, List

# --- 模拟工具集 ---
def calculate(expression: str) -> str:
    try:
        # 仅允许安全数学字符
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            return str(eval(expression))
        return "Error: 包含非法字符"
    except Exception as e:
        return f"Error: {str(e)}"

def get_current_date(param: str = "") -> str:
    return "2026-07-17"

TOOLS = {
    "calculate": calculate,
    "get_current_date": get_current_date
}
# --- 1. 动态记忆管理器 (Memory Manager) ---

class SlidingWindowMemory:
    """滑动窗口记忆管理器，防止 Observation 撑爆上下文"""
    def __init__(self, max_rounds: int = 3, max_obs_len: int = 300):
        self.max_rounds = max_rounds       # 最多保留最近的几轮 Thought-Action-Observation 对
        self.max_obs_len = max_obs_len     # 单个 Observation 的最大字符长度，超出则截断
        self.history: List[Dict[str, str]] = []

    def add_step(self, thought: str, action: str, observation: str):
        # 截断过长的 Observation，防止爆 Context
        if len(observation) > self.max_obs_len:
            observation = observation[:self.max_obs_len] + "... [此处数据由于过长已被系统截断] ..."

        self.history.append({
            "thought": thought,
            "action": action,
            "observation": observation
        })

        # 维持滑动窗口大小
        if len(self.history) > self.max_rounds:
            self.history.pop(0)

    def get_formatted_history(self) -> str:
        """格式化输出历史记忆，组装成 Prompt"""
        formatted = ""
        for i, step in enumerate(self.history):
            formatted += f"\n(历史第 {i+1} 步)\n"
            formatted += f"Thought: {step['thought']}\n"
            formatted += f"Action: {step['action']}\n"
            formatted += f"Observation: {step['observation']}\n"
        return formatted

# --- 2. 具备自愈能力的解析器 (Self-Healing Parser) ---
class RobustParser:
    @staticmethod
    def parse_action(llm_output: str) -> Dict[str, Any]:
        """
        鲁棒的解析器：尝试多种正则匹配。
        如果解析彻底失败，抛出 ValueError，说明具体的格式错误。
        """
        # 兼容标准格式: Action: calculate[2+2]
        match = re.search(r"Action:\s*(\w+)\[(.*?)\]", llm_output)
        if match:
            return {"tool": match.group(1), "args": match.group(2)}

        # 兼容模型误写的圆括号格式: Action: calculate(2+2)
        match_paren = re.search(r"Action:\s*(\w+)\((.*?)\)", llm_output)
        if match_paren:
            return {"tool": match_paren.group(1), "args": match_paren.group(2)}

        # 检查是否包含 Final Answer
        if "Final Answer:" in llm_output:
            final_ans = llm_output.split("Final Answer:")[-1].strip()
            return {"final_answer": final_ans}

        # 格式完全错误，抛出异常（该异常会被捕获并作为 Observation 反馈给模型以实现自愈）
        raise ValueError(
            "未能解析出合法的 Action 或 Final Answer。请确保你的输出格式严格遵循:\n"
            "Thought: 你的思考过程\n"
            "Action: 工具名称[参数]\n"
            "注意：不要把方括号写成圆括号，且一次只能执行一个 Action。"
        )

# --- 3. 升级版模拟大模型 ---
class MockFaultyLLM:
    """模拟一个容易出错的大模型，验证 Agent 的自愈能力"""
    def __init__(self):
        self.step = 0

    def generate(self, full_prompt: str) -> str:
        self.step += 1
        if self.step == 1:
            # 模拟模型犯错：使用了圆括号，且没有写 Thought
            return "Action: calculate(500 * 2.5)"
        elif self.step == 2:
            # 观察到自愈提示后，模型改正了错误，并给出了标准输出
            return "Thought: 抱歉，上一步我格式输错了。现在我重新计算。\nAction: calculate[500 * 2.5]"
        else:
            return "Thought: 计算完成，结果是1250。\nFinal Answer: 最终计算结果为 1250。"

# --- 4. 容错控制引擎核心循环 ---
def run_resilient_agent(user_query: str):
    llm = MockFaultyLLM()
    memory = SlidingWindowMemory(max_rounds=3, max_obs_len=150)

    system_prompt = """你是一个智能 Agent。你可用的工具有：
1. calculate[expression]: 计算数学题
2. get_current_date[]: 获取日期

每次你必须按照以下格式输出：
Thought: 思考过程
Action: 工具名[参数]

一旦得到最终答案，必须输出：
Final Answer: 你的最终回答
"""

    current_thought = ""
    current_action = ""
    max_steps = 6

    print(f"🚀 【Resilient Agent 启动】 任务: {user_query}\n")
    for step_idx in range(max_steps):
        print(f"\n--- 🔄 [Step {step_idx + 1}] ---")

        # 1. 组装 Prompt (系统级提示词 + 滚动记忆窗口 + 当前用户查询)
        prompt = system_prompt + "\n=== 历史记忆窗口 ===" + memory.get_formatted_history()
        prompt += f"\nUser: {user_query}\n"

        # 2. 调用模型生成
        llm_output = llm.generate(prompt)
        print(f"🤖 [LLM 输出]:\n{llm_output}")

        # 提取当前生成的 Thought (简单切分作为记录)
        thought_part = llm_output.split("Action:")[0].replace("Thought:", "").strip()
        current_thought = thought_part if thought_part else "进行下一步操作"

        # 3. 解析与自愈捕获
        try:
            parsed = RobustParser.parse_action(llm_output)

            # 命中 Final Answer，任务圆满结束
            if "final_answer" in parsed:
                print(f"\n🎯 【任务达成】 Final Answer: {parsed['final_answer']}")
                break

            current_action = f"{parsed['tool']}[{parsed['args']}]"

            # 4. 执行工具
            tool_name = parsed["tool"]
            tool_arg = parsed["args"]

            if tool_name in TOOLS:
                print(f"🔧 [执行工具] 调用 {tool_name} -> 参数: '{tool_arg}'")
                observation = TOOLS[tool_name](tool_arg)
            else:
                observation = f"Error: 工具 {tool_name} 未定义。"

        except ValueError as e:
            # 【核心容错逻辑】：如果解析失败，捕获异常
            # 我们将具体的报错信息伪装成一个 Observation 送给模型，迫使其在下一步进行“格式自愈”！
            error_message = str(e)
            print(f"⚠️ [格式解析失败]: 触发自愈机制。向模型反馈错误信息...")
            current_action = "格式损坏的Action"
            observation = f"System Error (格式错误): {error_message}"

        print(f"👁️ [获得观察]: {observation}")

        # 5. 将这一步的产物塞入滑动记忆窗口
        memory.add_step(current_thought, current_action, observation)

if __name__ == "__main__":
    run_resilient_agent("计算 500 乘以 2.5 是多少？")

1. 动手实验与日志观察：
   * 运行上述自愈 Agent 代码。仔细观察 Step 1 到 Step 2 之间 Prompt 发生的奇妙变化。
   * 思考：为什么我们不需要手动写条件语句去纠正模型的圆括号，仅靠把 System Error (格式错误): ... 作为 Observation 喂给模型，它就能在第二步自行改正？这利用了 LLM 的什么本质能力？

2. 长期记忆与知识沉淀的引入：
   * `SlidingWindowMemory` 虽然通过“丢弃旧步骤”保住了上下文不溢出，但这也带来了一个致命缺陷：一旦解决任务的步骤超过了滑动窗口大小（例如大于3步），Agent 就会忘记它在第 1 步和第 2 步已经探索过的路线，从而导致逻辑上的“鬼打墙”。
   * 工程挑战：在下周引入 RAG（检索增强生成）之前，请你想一想，如果不使用向量数据库，我们能不能让大模型在每一步执行完后，自动维护一份“全局备忘录（Global Scratchpad / State Summary）”？每次只把这个高浓缩的“备忘录”和最近一轮的 Observation 传给模型。你会如何设计这个备忘录的 Prompt 格式？